# Lab 9d: Skip-Gram Word Embeddings with Full Softmax and Negative Sampling

In this lab, we train a Skip-Gram model on the same chemistry corpus used in L9b. We first train with the full softmax objective, then implement Negative Sampling and compare the training dynamics. Finally, we compare the learned Skip-Gram embeddings to Continuous Bag of Words (CBOW) embeddings trained on the same data.

> __Learning Objectives__
>
> By the end of this lab, you should be able to:
> * __Generate Skip-Gram training pairs from a domain corpus:__ Build (target, context) training pairs from the chemistry corpus and compare the number of pairs to CBOW for the same window size. Explain why Skip-Gram produces more training signal per word occurrence.
> * __Train Skip-Gram with full softmax and Negative Sampling:__ Train a Skip-Gram model using both the full softmax cross-entropy objective and the Negative Sampling binary classification objective. Compare the loss curves and convergence behavior of each approach.
> * __Compare Skip-Gram and CBOW embeddings:__ Extract word embeddings from Skip-Gram and CBOW models trained on the same corpus and compare cosine similarities for selected word pairs. Evaluate whether Skip-Gram captures different word relationships than CBOW.

Let's get started!

___

<div>
  <center>
    <img src="figs/Fig-SkipGram-Context-Window.svg" style="width:75%" />
  </center>
</div>

## Background: Skip-Gram and Negative Sampling

The Skip-Gram model reverses the CBOW prediction direction: instead of predicting a target word from its context, Skip-Gram predicts each context word from the target word. This produces multiple training pairs per word position, giving the model more training signal per sentence.

> __Definition (Skip-Gram Model):__
>
> Let $\mathcal{V}$ be a vocabulary of size $N_{\mathcal{V}}$, $d_h \in \mathbb{Z}_{>0}$ the embedding dimension, and $m$ the context window half-width. For each target word at position $t$ in a sentence, the Skip-Gram model produces $2m$ training pairs, one for each context word within positions $[t-m, t+m]$. The forward pass for a single (target, context) pair proceeds in __three steps__:
>
> 1. The target word is represented as a one-hot vector $\mathbf{x} \in \{0,1\}^{N_{\mathcal{V}}}$ and projected into the embedding space: $\mathbf{h} = \mathbf{W}_{1}\,\mathbf{x} \in \mathbb{R}^{d_h}$, where $\mathbf{W}_{1} \in \mathbb{R}^{d_h \times N_{\mathcal{V}}}$.
> 2. The hidden representation is mapped to output logits: $\mathbf{u} = \mathbf{W}_{2}\,\mathbf{h} \in \mathbb{R}^{N_{\mathcal{V}}}$, where $\mathbf{W}_{2} \in \mathbb{R}^{N_{\mathcal{V}} \times d_h}$.
> 3. The logits are normalized by softmax to give a predicted probability for each vocabulary word:
> $$\hat{y}_i = \frac{e^{u_i}}{\displaystyle\sum_{j=1}^{N_{\mathcal{V}}} e^{u_j}}, \quad i = 1,\dots,N_{\mathcal{V}}$$
> The model is trained by minimizing the cross-entropy loss $\mathcal{L} = -\sum_{i=1}^{N_{\mathcal{V}}} y_i \log \hat{y}_i$, where $\mathbf{y} \in \{0,1\}^{N_{\mathcal{V}}}$ is the one-hot vector of the context word. After training, the columns of $\mathbf{W}_{1}$ are the learned word embeddings.

The full softmax normalization sums over the entire vocabulary at every update, making it expensive for large vocabularies. Negative Sampling replaces this with a binary classification objective.

> __Definition (Negative Sampling Objective):__
>
> For a target embedding $\mathbf{h} = \mathbf{W}_{1}[:,t]$ and context word embedding $\mathbf{w}_c = \mathbf{W}_{2}[c,:]^{\top}$, the Negative Sampling loss for one (target, context) pair is:
>
> $$\mathcal{L} = \underbrace{-\log\sigma(\mathbf{w}_c^{\top}\mathbf{h})}_{\text{positive term}} + \underbrace{\sum_{i=1}^{k}\left[-\log\sigma(-\mathbf{w}_{n_i}^{\top}\mathbf{h})\right]}_{\text{negative term}}$$
>
> where $\sigma(\cdot)$ is the sigmoid function, $k$ is the number of negative samples, and each $n_i$ is drawn from the noise distribution $P_n(w) \propto f(w)^{3/4}$, where $f(w)$ is the corpus frequency of word $w$. This reduces the per-update cost from $O(N_{\mathcal{V}})$ to $O(k)$.

___

## Setup, Data, and Prerequisites

We set up the computational environment by including the `Include.jl` file, loading any needed resources, such as sample datasets, and setting up any required constants.

> __Environment Setup with Include.jl__
>
> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, and includes local source files in `src/`. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [1]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material.

### Implementations

The notebook uses the following functions:

| Function | Location | Description |
|----------|----------|-------------|
| `build_skipgram_pairs(sentences, vocabulary; window_size)` | `src/SkipGram.jl` | Generates one (target, context) pair per target-context word combination within the window |
| `train_skipgram_softmax(training_pairs, vocab_size; d_h, eta, num_epochs)` | `src/SkipGram.jl` | Trains Skip-Gram with full softmax cross-entropy. Returns `W1`, `W2`, and `loss_history` |
| `build_noise_distribution(sentences, vocabulary)` | `src/NegativeSampling.jl` | Builds the smoothed unigram noise distribution $P_n(w) \propto f(w)^{3/4}$ |
| `train_skipgram_ns(training_pairs, vocab_size, noise_dist; d_h, eta, num_epochs, k)` | `src/NegativeSampling.jl` | Trains Skip-Gram with Negative Sampling. Returns `W1`, `W2`, and `loss_history` |
| `build_cbow_pairs(sentences, vocabulary; window_size)` | `src/CBOW.jl` | Generates (context_sum, target) training pairs for CBOW |
| `train_cbow(training_pairs, vocab_size; d_h, eta, num_epochs)` | `src/CBOW.jl` | Trains a CBOW model via SGD. Returns `W1`, `W2`, and `loss_history` |

### Constants

Let's set some constants that we will use later. Please look at the comments in the code for more details on each constant's permissible values, units, etc.

In [ ]:
window_size = 3;     # context window half-width on each side of the target word
d_h         = 30;    # embedding dimension (hidden layer size)
eta         = 0.01;  # SGD learning rate
num_epochs  = 200;   # number of training epochs
k           = 5;     # number of negative samples per positive pair

### Data

We load the same 767-sentence chemistry corpus used in L9b from `data/chemistry-corpus.txt`. This corpus encodes semantic relationships between concept pairs such as `acid`/`base`, `oxidation`/`reduction`, `exothermic`/`endothermic`, and `solid`/`liquid`/`gas` through parallel sentence structure.

> __What is going on in this code block?__
>
> We read each line of the corpus text file [using the `readlines(...)` function](https://docs.julialang.org/en/v1/base/io-network/#Base.readlines) and discard any empty lines [using the `filter(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.filter). The `sentences::Vector{String}` variable holds the loaded corpus sentences.

Let's load the corpus:

In [3]:
sentences = let
    path = joinpath(_PATH_TO_DATA, "chemistry-corpus.txt")
    filter(!isempty, readlines(path))
end;
println("Corpus: $(length(sentences)) sentences")

Corpus: 767 sentences


> __What is going on in this code block?__
>
> We tokenize each sentence by converting to lowercase and splitting on whitespace [using the `split(...)` function](https://docs.julialang.org/en/v1/base/strings/#Base.split). We concatenate the token lists across all sentences [using the `vcat(...)` function](https://docs.julialang.org/en/v1/base/arrays/#Base.vcat) and deduplicate [using the `unique(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.unique). Four control tokens (`<bos>`, `<eos>`, `<pad>`, `<unk>`) are reserved at the start of the index. The `vocabulary::Dict{String,Int64}` variable holds the token-to-index mapping, and the `inverse_vocabulary::Dict{Int64,String}` variable holds the index-to-token mapping.

Let's build the vocabulary:

In [4]:
vocabulary, inverse_vocabulary = let
    control_tokens = ["<bos>", "<eos>", "<pad>", "<unk>"];
    all_words = vcat([split(lowercase(s)) for s in sentences]...);
    unique_words = unique(all_words);
    vocab = Dict{String,Int64}();
    for (i, token) in enumerate(control_tokens)
        vocab[token] = i;
    end
    offset = length(control_tokens);
    for (i, word) in enumerate(unique_words)
        if !haskey(vocab, word)
            vocab[word] = offset + i;
        end
    end
    inv_vocab = Dict{Int64,String}(v => k for (k, v) in vocab);
    println("Vocabulary size: $(length(vocab)) tokens");
    vocab, inv_vocab
end;

Vocabulary size: 1563 tokens


___

## Task 1: Build Skip-Gram Training Pairs

Skip-Gram reverses the CBOW prediction direction. For each target word at position $t$ in a sentence, we generate one (target, context) pair for every context word within the window $[t - m, t + m]$, where $m$ is the window half-width. This produces up to $2m$ training pairs per word occurrence, compared to one pair per word in CBOW.

> __What is going on in this code block?__
>
> We generate Skip-Gram training pairs [using the `build_skipgram_pairs(...)` function](src/SkipGram.jl) and CBOW training pairs [using the `build_cbow_pairs(...)` function](src/CBOW.jl) from the same corpus with the same window size. We compare the number of pairs produced by each method to verify that Skip-Gram generates more training signal. The `skipgram_pairs::Vector` variable holds the Skip-Gram (target, context) one-hot vector pairs, and the `cbow_pairs::Vector` variable holds the CBOW (context_sum, target) training pairs.

Let's generate the training pairs and compare:

In [5]:
skipgram_pairs = build_skipgram_pairs(sentences, vocabulary; window_size=window_size);
cbow_pairs = build_cbow_pairs(sentences, vocabulary; window_size=window_size);
println("Skip-Gram training pairs: $(length(skipgram_pairs))")
println("CBOW training pairs:      $(length(cbow_pairs))")
println("Ratio (SG/CBOW):          $(round(length(skipgram_pairs)/length(cbow_pairs), digits=2))")

Skip-Gram training pairs: 56514
CBOW training pairs:      10953
Ratio (SG/CBOW):          5.16


### Things to think about
* __Question:__ Why does Skip-Gram produce more training pairs than CBOW for the same window size? How does the ratio relate to the window half-width $m$?
* __Question:__ Each Skip-Gram pair contains a single target and a single context word, while each CBOW pair aggregates multiple context words into one input vector. What are the computational implications of this difference during training?

___

## Task 2: Train Skip-Gram with Full Softmax

We train the Skip-Gram model using the full softmax objective. At each update the model computes $\hat{\mathbf{y}} = \text{softmax}(\mathbf{W}_2 \mathbf{W}_1 \mathbf{x})$ and minimizes $-\log \hat{y}_c$, where $c$ is the index of the true context word. The cost per update is $O(N_{\mathcal{V}})$ because the softmax normalization sums over all vocabulary entries.

> __What is going on in this code block?__
>
> We fix the random seed for reproducibility [using the `Random.seed!(...)` function](https://docs.julialang.org/en/v1/stdlib/Random/#Random.seed!) and train the Skip-Gram model with full softmax [using the `train_skipgram_softmax(...)` function](src/SkipGram.jl). If a pre-trained model file exists on disk, we load it instead of retraining [using the `jldopen(...)` function](https://juliaio.github.io/JLD2.jl/dev/). The `W1_sg::Matrix{Float64}` variable holds the input weight matrix whose columns are the word embeddings, `W2_sg::Matrix{Float64}` holds the output weight matrix, and `loss_sg::Vector{Float64}` holds the average cross-entropy loss per epoch.

Let's train the model:

In [ ]:
W1_sg, W2_sg, loss_sg = let

    # initialize variables -
    W1_, W2_, lh_ = nothing, nothing, nothing;

    # look for a pre-trained model file; if exists, load; if not, train -
    model_path = joinpath(_PATH_TO_DATA, "skipgram_softmax_model.jld2");
    if isfile(model_path)
        println("Loading pre-trained Skip-Gram (softmax) model from $(model_path)");
        f = jldopen(model_path, "r");
        W1_ = f["W1"]; W2_ = f["W2"]; lh_ = f["loss_history"];
        close(f);
        println("Model loaded successfully!");
    else
        println("Training Skip-Gram (softmax) model ($(num_epochs) epochs, $(length(skipgram_pairs)) pairs)...");
        Random.seed!(42);
        W1_, W2_, lh_ = train_skipgram_softmax(skipgram_pairs, length(vocabulary);
            d_h=d_h, eta=eta, num_epochs=num_epochs);
        
        jldsave(model_path; W1=W1_, W2=W2_, loss_history=lh_);
        println("Model saved to $(model_path)");
    end

    W1_, W2_, lh_
end;

Let's plot the loss curve to verify that training converged:

In [ ]:
plot(loss_sg; xlabel="Epoch", ylabel="Average Cross-Entropy Loss",
    title="Skip-Gram (Full Softmax) Training Loss", legend=false, lw=2, framestyle=:box)

### Things to think about
* __Question:__ Look at the loss curve. Does it flatten out by the end of training, or is it still decreasing? If it were still dropping, would you increase the number of epochs, increase the learning rate, or increase the embedding dimension?
* __Question:__ The full softmax computes a probability distribution over the entire vocabulary at every update. For a vocabulary of size $N_{\mathcal{V}}$, what is the computational cost per training pair? How would this scale if you trained on a corpus with 100,000 unique words?

___

## Task 3: Train Skip-Gram with Negative Sampling

Instead of computing the full softmax over the vocabulary, Negative Sampling uses a binary classification objective. For each positive (target, context) pair, we sample $k$ noise words from a smoothed unigram distribution and train the model to score the positive pair higher than the noise pairs using sigmoid activations.

> __What is going on in this code block?__
>
> We build the noise distribution from corpus word frequencies [using the `build_noise_distribution(...)` function](src/NegativeSampling.jl). Each entry is the sampling probability for the corresponding word, computed as $P_n(w) \propto f(w)^{3/4}$, where $f(w)$ is the word count. The `noise_distribution::Vector{Float64}` variable holds the sampling probability for each vocabulary index.

The $3/4$ exponent in the noise distribution deserves some explanation.

> __Why the $3/4$ exponent?:__
>
> Sampling negatives proportional to raw frequency $f(w)$ would over-represent common words (e.g., `the`, `is`) and under-represent rare but meaningful words (e.g., `catalyst`, `endothermic`). Raising frequencies to the $3/4$ power compresses the distribution: it reduces the sampling probability of frequent words and increases the relative probability of rare words. 
> 
> For example, if a common word is 100x more frequent than a rare word under raw frequency, that ratio drops to roughly 32x after the $3/4$ smoothing. The exponent $\alpha = 3/4$ sits between two extremes: $\alpha = 1$ (raw frequency, biased toward common words) and $\alpha = 0$ (uniform, ignores frequency entirely). The value $3/4$ was found empirically to give the best performance on analogy tasks in the original Word2Vec paper (Mikolov et al., 2013).

Let's build the noise distribution:

In [ ]:
noise_distribution = build_noise_distribution(sentences, vocabulary);
println("Noise distribution: $(length(noise_distribution)) entries, sums to $(round(sum(noise_distribution), digits=4))")

### Worked Example: Negative Sampling Loss for One Training Pair

Before training, let's trace the Negative Sampling loss for a single (target, context) pair to clarify how the signs work. Recall the loss from the Background section:

$$\mathcal{L} = \underbrace{-\log\sigma(s_c)}_{\text{positive term}} + \underbrace{\sum_{i=1}^{k}\left[-\log\sigma(-s_{n_i})\right]}_{\text{negative terms}}$$

where $\sigma(x) = 1/(1+e^{-x})$ is the sigmoid function, $s_c = (\mathbf{w}_c^{(2)})^{\top}\mathbf{h}$ is the dot product between the true context embedding and the target embedding, and $s_{n_i} = (\mathbf{w}_{n_i}^{(2)})^{\top}\mathbf{h}$ is the dot product for the $i$-th negative sample. Suppose the target word is `acid`, the true context word is `base`, and we draw $k = 1$ negative sample: `boiling`. The current dot products are $s_c = +2.0$ and $s_n = +1.5$.

> __Positive term:__
>
> We feed the dot product as-is into $\sigma$: $-\log\sigma(+2.0) = -\log\!\left(\frac{1}{1+e^{-2.0}}\right) = -\log(0.881) = 0.127$. This is small because $s_c$ is large and positive, so the model already places `base` near `acid` in embedding space.

Now consider the negative sample.

> __Negative term:__
>
> We feed the **negated** dot product into $\sigma$: $-\log\sigma(-1.5) = -\log\!\left(\frac{1}{1+e^{+1.5}}\right) = -\log(0.182) = 1.704$. This is large because $s_n = +1.5$ means `boiling` is still close to `acid`, and the model is penalized for this.

The total loss is $0.127 + 1.704 = 1.831$. Gradient descent reduces this loss by pushing $s_c$ more positive (moving `base` closer to `acid`) and pushing $s_n$ more negative (moving `boiling` away from `acid`). The following summary explains how the sign convention produces these opposing effects from a single loss structure.

> __Why the signs work:__
>
> Both terms share the outer structure $-\log\sigma(\cdot)$, which is minimized when its argument is large and positive. The sign flip happens **inside** $\sigma$: the positive term receives $+s_c$, so minimization pushes $s_c$ large and positive (words close together). The negative term receives $-s_n$, so minimization pushes $-s_n$ large and positive, which means $s_n$ becomes large and negative (words far apart). The negation inside $\sigma(-s_n)$ is what converts "minimize this term" from "push together" to "push apart."

Now let's train the model on the full corpus using this objective.

> __What is going on in this code block?__
>
> We fix the random seed and train the Skip-Gram model with Negative Sampling [using the `train_skipgram_ns(...)` function](src/NegativeSampling.jl). For each positive (target, context) pair, the model samples $k = 5$ negative words from the noise distribution and updates only those $k + 1$ output embeddings instead of the full vocabulary. If a pre-trained model file exists on disk, we load it instead of retraining. The `W1_ns::Matrix{Float64}` variable holds the input weight matrix whose columns are the word embeddings, `W2_ns::Matrix{Float64}` holds the output weight matrix, and `loss_ns::Vector{Float64}` holds the average Negative Sampling loss per epoch.

Let's train the model:

In [ ]:
W1_ns, W2_ns, loss_ns = let

    # initialize variables -
    W1_, W2_, lh_ = nothing, nothing, nothing;

    # look for a pre-trained model file; if exists, load; if not, train -
    model_path = joinpath(_PATH_TO_DATA, "skipgram_ns_model.jld2");
    if isfile(model_path)
        println("Loading pre-trained Skip-Gram (Negative Sampling) model from $(model_path)");
        f = jldopen(model_path, "r");
        W1_ = f["W1"]; W2_ = f["W2"]; lh_ = f["loss_history"];
        close(f);
        println("Model loaded successfully!");
    else
        println("Training Skip-Gram (Negative Sampling) model ($(num_epochs) epochs, $(length(skipgram_pairs)) pairs, k=$(k))...");
        Random.seed!(42);
        W1_, W2_, lh_ = train_skipgram_ns(skipgram_pairs, length(vocabulary), noise_distribution;
            d_h=d_h, eta=eta, num_epochs=num_epochs, k=k);
        
        jldsave(model_path; W1=W1_, W2=W2_, loss_history=lh_);
        println("Model saved to $(model_path)");
    end

    W1_, W2_, lh_
end;

Let's compare the training loss curves for both approaches:

In [ ]:
let
    p1 = plot(loss_sg; label="Full Softmax", lw=2, xlabel="Epoch", ylabel="Average Loss",
        title="Skip-Gram (Full Softmax)", legend=:topright, framestyle=:box);
    p2 = plot(loss_ns; label="Negative Sampling (k=$(k))", lw=2, xlabel="Epoch", ylabel="Average Loss",
        title="Skip-Gram (Negative Sampling)", legend=:topright, framestyle=:box, color=:orange);
    plot(p1, p2; layout=(1,2), size=(900, 400))
end

### Things to think about
* __Question:__ The two loss curves use different objective functions (cross-entropy vs. binary cross-entropy), so their magnitudes are not directly comparable. What does each curve tell you about convergence?
* __Question:__ Negative Sampling updates only $k + 1$ output embeddings per training pair, while full softmax updates all $N_{\mathcal{V}}$ output embeddings. For this corpus, what is the speedup factor? Which approach would scale better to a vocabulary of 100,000 words?

___

## Task 4: Compare Skip-Gram and CBOW Embeddings

To compare with CBOW, we train a CBOW model on the same corpus and then compute cosine similarities for selected word pairs across all three models: CBOW, Skip-Gram (full softmax), and Skip-Gram (Negative Sampling). Embeddings are the columns of $\mathbf{W}_1$ in each model.

> __Why compare?:__
>
> CBOW and Skip-Gram optimize different objectives on the same data. CBOW predicts a target from context (averaging over context words), while Skip-Gram predicts each context word from the target. We expect both to capture co-occurrence patterns, but they may produce different similarity rankings for word pairs that vary in frequency or context diversity.

We start by training a CBOW model on the same corpus so the comparison is controlled.

> __What is going on in this code block?__
>
> We fix the random seed and train a CBOW model on the same corpus [using the `train_cbow(...)` function](src/CBOW.jl). If a pre-trained model file exists on disk, we load it instead of retraining. The `W1_cbow::Matrix{Float64}` variable holds the input weight matrix whose columns are the CBOW word embeddings, `W2_cbow::Matrix{Float64}` holds the output weight matrix, and `loss_cbow::Vector{Float64}` holds the average cross-entropy loss per epoch.

First, let's train a CBOW model on the same corpus:

In [ ]:
W1_cbow, W2_cbow, loss_cbow = let

    # initialize variables -
    W1_, W2_, lh_ = nothing, nothing, nothing;

    # look for a pre-trained model file; if exists, load; if not, train -
    model_path = joinpath(_PATH_TO_DATA, "cbow_model.jld2");
    if isfile(model_path)
        println("Loading pre-trained CBOW model from $(model_path)");
        f = jldopen(model_path, "r");
        W1_ = f["W1"]; W2_ = f["W2"]; lh_ = f["loss_history"];
        close(f);
        println("Model loaded successfully!");
    else
        println("Training CBOW model ($(num_epochs) epochs, $(length(cbow_pairs)) pairs)...");
        Random.seed!(42);
        W1_, W2_, lh_ = train_cbow(cbow_pairs, length(vocabulary);
            d_h=d_h, eta=eta, num_epochs=num_epochs, verbose=false);
        
        jldsave(model_path; W1=W1_, W2=W2_, loss_history=lh_);
        println("Model saved to $(model_path)");
    end

    W1_, W2_, lh_
end;

> __What is going on in this code block?__
>
> We define a helper function `cosine_sim` that retrieves embedding vectors for two words from a weight matrix and computes their cosine similarity. We then evaluate cosine similarity for chemistry-themed word pairs across all three models and display the results in a table.

Let's compare cosine similarities across all three models:

In [ ]:
let
    # helper: cosine similarity between two word embeddings from W
    function cosine_sim(W::Matrix{Float64}, w1::String, w2::String, vocab::Dict{String, Int64})
        i = get(vocab, w1, 0);
        j = get(vocab, w2, 0);
        (i == 0 || j == 0) && return NaN;
        v1 = W[:, i]; v2 = W[:, j];
        return dot(v1, v2) / (norm(v1) * norm(v2) + 1e-10);
    end

    # word pairs from the chemistry domain -
    word_pairs = [
        ("acid", "base"),
        ("oxidation", "reduction"),
        ("exothermic", "endothermic"),
        ("solid", "liquid"),
        ("catalyst", "reaction"),
        ("acid", "gas"),
    ];

    # build results table -
    header = ["Word 1", "Word 2", "CBOW", "SG Softmax", "SG Neg. Samp."];
    rows = [];
    for (w1, w2) in word_pairs
        c  = round(cosine_sim(W1_cbow, w1, w2, vocabulary), digits=4);
        sg = round(cosine_sim(W1_sg,   w1, w2, vocabulary), digits=4);
        ns = round(cosine_sim(W1_ns,   w1, w2, vocabulary), digits=4);
        push!(rows, [w1, w2, c, sg, ns]);
    end

    pretty_table(hcat(rows...)'; header=header,
        backend = :text,
        table_format = TextTableFormat(borders = text_table_borders__compact)
    );
end

Each cell is the cosine similarity between two word embeddings from that model, ranging from $-1$ (opposite directions in embedding space) to $1$ (same direction). Values near $0$ indicate no learned relationship.

> __Reading the table:__
>
> * **`acid / base` and `oxidation / reduction`** are concept pairs that appear in parallel sentence structures in the corpus, so all three models should capture their semantic similarity.
> * **`exothermic / endothermic`** and **`solid / liquid`** test whether the models learn that opposing concepts within a category share context.
> * **`acid / gas`** is a cross-category sanity check. These words belong to different semantic groups and should show lower similarity than within-category pairs.

### Things to think about
* __Question:__ Do the three models agree on which word pairs are most similar? Where do they disagree most, and what might explain the differences?
* __Question:__ The `acid / gas` pair serves as a negative control. If this pair scored high in one model, what would that suggest about the quality of that model's embeddings?

___

## Summary

This lab trained Skip-Gram models using both full softmax and Negative Sampling on a chemistry corpus, and compared the resulting embeddings to CBOW trained on the same data.

> __Key Takeaways__
>
> * **Skip-Gram generates more training pairs than CBOW:** For window half-width $m$, each word occurrence produces up to $2m$ (target, context) pairs, giving Skip-Gram more training signal per sentence than CBOW. This reversal of input and output is the structural difference between the two architectures.
> * **Negative Sampling replaces full softmax with binary classification:** Instead of normalizing over the full vocabulary at cost $O(N_{\mathcal{V}})$, Negative Sampling trains the model to distinguish true context words from $k$ noise samples at cost $O(k)$. Both objectives converge on this corpus, but Negative Sampling scales to larger vocabularies.
> * **Different training objectives produce different similarity rankings:** CBOW and Skip-Gram capture co-occurrence patterns from the same data but weight them differently because they optimize different objectives. Comparing cosine similarities across models reveals which word relationships each architecture captures most strongly.

The differences between models become more pronounced on larger corpora where word frequency distributions are broader and context diversity is higher.

___